# Notebook 1:
### Synthetic Data Generation and Core Dataset Construction

**Purpose**

This notebook generates the core synthetic datasets used throughout the pharmacy operations optimization project. It creates realistic, schema-consistent data that simulates prescription activity, inventory levels, sales transactions, patient mix, and pharmacist staffing patterns.

#### Outputs
- `patients.csv`
- `products.csv`
- `prescriptions.csv`
- `inventory.csv`
- `sales.csv`
- `staffschedule.csv`
-  `ref_drugs.csv`
-  `prescriptions_by_class.csv`

**Key responsibilities**
- Builds internally consistent relational data
- Aligns staffing weeks to prescription demand range
- Injects realistic variability into prescription volume and staffing hours
- Ensures downstream notebooks have clean, reproducible inputs

This notebook establishes the foundational data layer that feeds forecasting, staffing optimization, and Power BI analytics.

In [12]:
# --- Config (paths) ---
from pathlib import Path

PROJECT_ROOT = Path(".").resolve()
DATA_DIR = PROJECT_ROOT
OUT_DIR = PROJECT_ROOT / "bi_outputs"
FORECAST_DIR = PROJECT_ROOT / "forecast"

OUT_DIR.mkdir(parents=True, exist_ok=True)
FORECAST_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT)
print("OUT_DIR       =", OUT_DIR)
print("FORECAST_DIR  =", FORECAST_DIR)


PROJECT_ROOT = /Users/selenadavis/PythonProject1/PharmacyOperationsOptimization/notebooks
OUT_DIR       = /Users/selenadavis/PythonProject1/PharmacyOperationsOptimization/notebooks/bi_outputs
FORECAST_DIR  = /Users/selenadavis/PythonProject1/PharmacyOperationsOptimization/notebooks/forecast


In [13]:
# --- Setup ---
from pathlib import Path
import pandas as pd
import numpy as np


In [14]:
# --- Helpers ---
def rename_any(df: pd.DataFrame, candidates: list[str], canonical: str) -> pd.DataFrame:
    """Rename the first matching candidate column to canonical (if canonical doesn't already exist)."""
    for c in candidates:
        if c in df.columns and canonical not in df.columns:
            return df.rename(columns={c: canonical})
    return df

def require_cols(df: pd.DataFrame, name: str, cols: list[str]) -> None:
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"{name} missing columns: {missing}. Found: {list(df.columns)}")

def to_num(s):
    return pd.to_numeric(s, errors="coerce")

def norm_str(s):
    return pd.Series(s, dtype="string").astype("string").str.strip()

In [15]:
# --- Synthetic data generator ---
def build_synthetic_data(files: dict, write_only=None, seed: int = 42) -> None:
    """
    Create schema-compatible synthetic CSVs.

    write_only: set of keys to write (subset of files dict keys).
               Default writes all core datasets.
    """
    if write_only is None:
        write_only = {"patients", "prescriptions", "products", "inventory", "sales", "staffschedule"}
    else:
        write_only = set(write_only)

    rng = np.random.default_rng(seed)

    # Patients
    n_patients = 400
    patients = pd.DataFrame({
        "PatientID": [f"P{str(i).zfill(5)}" for i in range(1, n_patients + 1)],
        "InsuranceType": rng.choice(["Commercial", "Medicare", "Medicaid", "Cash"], size=n_patients,
                                    p=[0.35, 0.30, 0.25, 0.10]),
    })

    # Products (drug catalog)
    n_products = 220
    a = [str(i).zfill(4) for i in range(1, n_products + 1)]
    b = [str(i).zfill(4) for i in range(1, n_products + 1)]
    c = [str((i * 11) % 100).zfill(2) for i in range(1, n_products + 1)]
    drug_ids = [f"{aa}-{bb}-{cc}" for aa, bb, cc in zip(a, b, c)]

    base_names = rng.choice(
        ["Metformin", "Lisinopril", "Atorvastatin", "Amlodipine", "Levothyroxine",
         "Gabapentin", "Sertraline", "Losartan", "Omeprazole", "Hydrochlorothiazide",
         "Amoxicillin", "Azithromycin", "Albuterol", "Insulin Glargine", "Fluoxetine",
         "Furosemide", "Prednisone", "Tamsulosin", "Warfarin", "Clopidogrel"],
        size=n_products,
        replace=True
    )
    strengths = rng.choice(["5 mg", "10 mg", "20 mg", "40 mg", "50 mg", "100 mg", "250 mg", "500 mg"], size=n_products,
                           replace=True)
    forms = rng.choice(["tablet", "capsule", "solution", "inhaler"], size=n_products, replace=True)
    drug_names = [f"{bn} {st} {fm}" for bn, st, fm in zip(base_names, strengths, forms)]

    products = pd.DataFrame({
        "DrugID": drug_ids,
        "DrugName": drug_names,
        "UnitCost": np.round(rng.uniform(0.05, 25.0, size=n_products), 2),
        "LeadTimeDays": rng.integers(1, 14, size=n_products),
    })

    # Prescriptions
    n_rx = 120000
    start = pd.Timestamp("2023-01-01")
    end = pd.Timestamp("2024-12-31")
    days_range = (end - start).days

    rx_dates = [start + pd.Timedelta(days=int(rng.integers(0, days_range + 1))) for _ in range(n_rx)]
    rx_drug = rng.choice(products["DrugID"].values, size=n_rx, replace=True)
    rx_patient = rng.choice(patients["PatientID"].values, size=n_rx, replace=True)

    prescriptions = pd.DataFrame({
        "RxID": [f"RX{str(i).zfill(7)}" for i in range(1, n_rx + 1)],
        "DrugID": rx_drug,
        "DrugName": pd.Series(rx_drug).map(products.set_index("DrugID")["DrugName"]).values,
        "QuantityDispensed": rng.integers(10, 120, size=n_rx),
        "RefillDate": pd.to_datetime(rx_dates),
        "Cost": np.round(rng.uniform(3.0, 250.0, size=n_rx), 2),
        "PatientID": rx_patient,
        "FillDate": pd.to_datetime(rx_dates),
    })
    prescriptions = prescriptions.merge(patients, on="PatientID", how="left")

    # Inventory snapshot
    suppliers = ["McKesson", "AmerisourceBergen", "Cardinal"]
    exp = (pd.Timestamp("2027-01-01") + pd.to_timedelta(rng.integers(30, 900, size=n_products), unit="D"))
    inventory = pd.DataFrame({
        "DrugID": products["DrugID"],
        "DrugName": products["DrugName"],
        "StockLevel": rng.integers(0, 2500, size=n_products),
        "ExpirationDate": exp.strftime("%m/%d/%y"),
        "Supplier": rng.choice(suppliers, size=n_products, replace=True),
        "DrugID_Original": products["DrugID"],
        "UnitCost": products["UnitCost"],
        "LeadTimeDays": products["LeadTimeDays"],
    })

    # Staff schedule (weekly, covers full Rx range)
    shifts = ["AM", "PM"]

    # Build week starts from prescription date range (Monday-based weeks)
    rx_dt = pd.to_datetime(prescriptions["FillDate"], errors="coerce").dt.normalize()
    rx_min = rx_dt.min()
    rx_max = rx_dt.max()
    week_start_min = (rx_min - pd.to_timedelta(rx_min.weekday(), unit="D")).normalize()
    week_start_max = (rx_max - pd.to_timedelta(rx_max.weekday(), unit="D")).normalize()
    week_starts = pd.date_range(week_start_min, week_start_max, freq="W-MON")

    # Weekly Rx volume for scaling
    tmp = prescriptions.copy()
    tmp["Date"] = pd.to_datetime(tmp["FillDate"], errors="coerce").dt.normalize()
    tmp["WeekStartDate"] = (tmp["Date"] - pd.to_timedelta(tmp["Date"].dt.weekday, unit="D")).dt.normalize()

    weekly_rx = (
        tmp.groupby("WeekStartDate", as_index=False)
           .size()
           .rename(columns={"size": "WeeklyRx"})
    )

    # Map for quick lookup
    rx_map = dict(zip(weekly_rx["WeekStartDate"], weekly_rx["WeeklyRx"]))

    # Synthetic staffing knobs that produce realistic hours
    RX_PER_STAFF_HOUR = 6.0
    COVERAGE_FLOOR_PER_DAY = 20.0
    PRODUCTIVE_FACTOR = 0.75
    AM_SHARE = 0.55
    PM_SHARE = 0.45

    # Roster: smaller, more realistic for synthetic data
    n_pharmacists = 8
    pharmacists = [f"RPH{str(i).zfill(3)}" for i in range(1, n_pharmacists + 1)]

    # Roster wages: stable per pharmacist
    wages = rng.normal(loc=62.0, scale=6.5, size=len(pharmacists)).clip(45, 85)
    wage_map = dict(zip(pharmacists, np.round(wages, 2)))

    # Per-shift minimum so you don’t get absurd tiny shifts
    MIN_SHIFT_HOURS = 6.0

    rows = []
    for ws in week_starts:
        ws_norm = pd.Timestamp(ws).normalize()
        weekly_rx_count = float(rx_map.get(ws_norm, weekly_rx["WeeklyRx"].median()))

        # Convert demand to required paid hours, then apply a floor
        required_staff_hours = weekly_rx_count / max(1e-6, RX_PER_STAFF_HOUR)
        floor_week = COVERAGE_FLOOR_PER_DAY * 7.0

        # Paid hours needed, adjusting for non-productive time
        total_week_hours = max(floor_week, required_staff_hours / max(1e-6, PRODUCTIVE_FACTOR))

        total_am = total_week_hours * AM_SHARE
        total_pm = total_week_hours * PM_SHARE

        # Spread hours across pharmacists with mild variation
        weights = rng.uniform(0.85, 1.15, size=len(pharmacists))
        weights = weights / weights.sum()

        am_hours = (total_am * weights) + rng.normal(0, 0.8, size=len(pharmacists))
        pm_hours = (total_pm * weights) + rng.normal(0, 0.8, size=len(pharmacists))

        # Clip small shifts but keep totals roughly stable by renormalizing
        am_hours = np.clip(am_hours, MIN_SHIFT_HOURS, None)
        pm_hours = np.clip(pm_hours, MIN_SHIFT_HOURS, None)

        # Renormalize back to target totals after clipping
        am_hours = am_hours * (total_am / am_hours.sum())
        pm_hours = pm_hours * (total_pm / pm_hours.sum())

        for i, pid in enumerate(pharmacists):
            rows.append({
                "PharmacistID": pid,
                "Shift": "AM",
                "HoursWorked": float(np.round(am_hours[i], 2)),
                "Wage": float(wage_map[pid]),
                "WeekStartDate": ws_norm.date().isoformat(),
            })
            rows.append({
                "PharmacistID": pid,
                "Shift": "PM",
                "HoursWorked": float(np.round(pm_hours[i], 2)),
                "Wage": float(wage_map[pid]),
                "WeekStartDate": ws_norm.date().isoformat(),
            })

    staffschedule = pd.DataFrame(rows)

    # Sales (transaction-level)
    sales = prescriptions[["RxID", "DrugID", "FillDate", "InsuranceType", "Cost"]].copy()
    sales = sales.sort_values("FillDate").reset_index(drop=True)
    sales["TransactionID"] = [f"T{str(i).zfill(8)}" for i in range(1, len(sales) + 1)]
    sales["Date"] = pd.to_datetime(sales["FillDate"], errors="coerce").dt.date.astype(str)
    sales["PaymentType"] = sales["InsuranceType"].fillna("Unknown").astype(str)
    sales["Revenue"] = pd.to_numeric(sales["Cost"], errors="coerce").fillna(0)
    sales = sales[["TransactionID", "DrugID", "Date", "PaymentType", "Revenue", "RxID"]]

    # Write outputs
    for p in files.values():
        Path(p).parent.mkdir(parents=True, exist_ok=True)

    if "patients" in write_only:
        patients.to_csv(files["patients"], index=False)
    if "products" in write_only:
        products.to_csv(files["products"], index=False)
    if "prescriptions" in write_only:
        prescriptions.to_csv(files["prescriptions"], index=False)
    if "inventory" in write_only:
        inventory.to_csv(files["inventory"], index=False)
    if "staffschedule" in write_only:
        staffschedule.to_csv(files["staffschedule"], index=False)
    if "sales" in write_only:
        sales.to_csv(files["sales"], index=False)

    print("Wrote:", sorted(write_only))

In [16]:
# --- Build core datasets ---
from pathlib import Path

BASE_DIR = Path(".").resolve()

FILES = {
    "patients": BASE_DIR / "patients.csv",
    "products": BASE_DIR / "products.csv",
    "prescriptions": BASE_DIR / "prescriptions.csv",
    "inventory": BASE_DIR / "inventory.csv",
    "sales": BASE_DIR / "sales.csv",
    "staffschedule": BASE_DIR / "staffschedule.csv",
}

FORCE_REBUILD_CORE = False
REBUILD_ONLY = None  # or set like {"staffschedule"}

In [18]:
# --- Schema ---
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR

DATA_DIR = REPO_ROOT / "data" / "generated"
BI_DIR   = REPO_ROOT / "bi_outputs"

print("Notebook dir:", NOTEBOOK_DIR)
print("Repo root:", REPO_ROOT)
print("Data dir:", DATA_DIR)
print("BI dir:", BI_DIR)

# --- Schema expectations ---
EXPECTED_GENERATED = {
    "patients.csv": ["PatientID", "InsuranceType"],
    "prescriptions.csv": ["RxID", "DrugID", "DrugName", "QuantityDispensed", "RefillDate", "Cost", "PatientID", "FillDate"],
    "inventory.csv": ["DrugID", "DrugName", "StockLevel", "ExpirationDate", "Supplier", "DrugID_Original", "UnitCost", "LeadTimeDays"],
    "sales.csv": ["TransactionID", "DrugID", "Date", "PaymentType", "Revenue", "RxID"],
    "staffschedule.csv": ["PharmacistID", "Shift", "HoursWorked", "Wage", "WeekStartDate"],
}

EXPECTED_BI_OUTPUTS = {
    "date_table.csv": ["Date"],
    "forecast_demand_by_class_day.csv": ["Date", "pharm_classes", "ForecastQty"],
    "prescription_volume_by_class_day.csv": ["Date", "pharm_classes", "Actual_Rx"],
    "prescription_volume_by_day.csv": ["Date", "TotalPrescriptions"],
    "projected_staffing_by_hour.csv": ["Date", "Hour", "RequiredStaff", "ScheduledStaff", "OverUnder", "OverUnderFlag"],
    "revenue_by_insurance_type.csv": ["InsuranceType", "Revenue", "RxCount"],
    "kpi_summary.csv": ["metric", "value", "unit", "source_file", "notes"],
}

def check_schema(path: Path, expected_cols: list[str]) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    df = pd.read_csv(path, nrows=5)
    missing = [c for c in expected_cols if c not in df.columns]
    if missing:
        raise ValueError(f"{path.name} missing columns: {missing}. Found: {list(df.columns)}")

# --- Run checks ---
print("\nChecking generated datasets...")
for fname, cols in EXPECTED_GENERATED.items():
    check_schema(DATA_DIR / fname, cols)

print("Generated schema checks passed.")

print("\nChecking Power BI import outputs...")
for fname, cols in EXPECTED_BI_OUTPUTS.items():
    check_schema(BI_DIR / fname, cols)

print("BI outputs schema checks passed.")

Notebook dir: /Users/selenadavis/PythonProject1/PharmacyOperationsOptimization/notebooks
Repo root: /Users/selenadavis/PythonProject1/PharmacyOperationsOptimization
Data dir: /Users/selenadavis/PythonProject1/PharmacyOperationsOptimization/data/generated
BI dir: /Users/selenadavis/PythonProject1/PharmacyOperationsOptimization/bi_outputs

Checking generated datasets...
Generated schema checks passed.

Checking Power BI import outputs...
BI outputs schema checks passed.


In [19]:
# ---vPaths & build controls ---
PROJECT_DIR = Path.cwd()

FILES = {
    "patients": PROJECT_DIR / "patients.csv",
    "prescriptions": PROJECT_DIR / "prescriptions.csv",
    "products": PROJECT_DIR / "products.csv",
    "inventory": PROJECT_DIR / "inventory.csv",
    "sales": PROJECT_DIR / "sales.csv",
    "staffschedule": PROJECT_DIR / "staffschedule.csv",
    "ref_drugs": PROJECT_DIR / "ref_drugs.csv",
    "prescriptions_by_class": PROJECT_DIR / "prescriptions_by_class.csv",
}

FORCE_REBUILD_CORE = False

REBUILD_ONLY = set()

FORCE_REBUILD_DERIVED = False

print("Project directory:", PROJECT_DIR)

Project directory: /Users/selenadavis/PythonProject1/PharmacyOperationsOptimization/notebooks


In [20]:
# --- Derived output: prescriptions_by_class.csv ---
import re

def norm_name(x):
    if pd.isna(x):
        return ""
    s = str(x).lower().strip()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^a-z0-9 ]+", "", s)
    s = re.sub(r"\b\d+(\.\d+)?\b", "", s)
    s = re.sub(r"\b(mg|mcg|g|ml|tab|tabs|cap|caps|unit|units|tablet|tablets|capsule|capsules|solution|inhaler)\b", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

if FILES["ref_drugs"].exists():
    rx = pd.read_csv(FILES["prescriptions"], low_memory=False)
    ref = pd.read_csv(FILES["ref_drugs"], low_memory=False)

    # detect date column
    date_col = "FillDate" if "FillDate" in rx.columns else "RefillDate"
    rx[date_col] = pd.to_datetime(rx[date_col], errors="coerce")

    # detect ref columns
    ref_cols_lower = {c.lower(): c for c in ref.columns}
    drugname_ref  = ref_cols_lower.get("drugname") or ref_cols_lower.get("name")
    drugclass_ref = ref_cols_lower.get("drugclass") or ref_cols_lower.get("class") or ref_cols_lower.get("pharm_classes")
    if not drugname_ref or not drugclass_ref:
        raise RuntimeError(f"ref_drugs.csv must include DrugName and DrugClass columns; found: {ref.columns.tolist()}")

    rx["DrugName_norm"] = rx["DrugName"].map(norm_name)
    ref["DrugName_norm"] = ref[drugname_ref].map(norm_name)

    name_map = (ref[["DrugName_norm", drugclass_ref]]
                .dropna(subset=["DrugName_norm"])
                .drop_duplicates(subset=["DrugName_norm"])
                .rename(columns={drugclass_ref: "DrugClass"}))

    rx2 = rx.merge(name_map, on="DrugName_norm", how="left")
    rx2["DrugClass"] = rx2["DrugClass"].fillna("Unmapped")
    rx2["QuantityDispensed"] = pd.to_numeric(rx2["QuantityDispensed"], errors="coerce").fillna(0)

    out = (rx2.dropna(subset=[date_col])
              .groupby([rx2[date_col].dt.date, "DrugName", "DrugClass"], as_index=False)
              .agg(Quantity=("QuantityDispensed", "sum"))
              .rename(columns={date_col: "Date"})
              .sort_values(["Date","DrugClass","DrugName"]))

    if FORCE_REBUILD_DERIVED or (not FILES["prescriptions_by_class"].exists()):
        out.to_csv(FILES["prescriptions_by_class"], index=False)
        print("Saved", FILES["prescriptions_by_class"].name, "shape", out.shape)
    else:
        print("prescriptions_by_class.csv already exists. Skipping derived rebuild.")
else:
    print("ref_drugs.csv not found. Skipping prescriptions_by_class.csv build.")

ref_drugs.csv not found. Skipping prescriptions_by_class.csv build.


In [22]:
# --- Repair utilities ---
def fix_missing_rxid_in_prescriptions(path="prescriptions.csv", backup=True):
    """Fill blank RxID values deterministically. Writes back to prescriptions.csv."""
    p = Path(path)
    rx = pd.read_csv(p, low_memory=False, dtype={"RxID":"string","DrugID":"string","PatientID":"string"})
    if backup:
        rx.to_csv(p.with_name(p.stem + "_backup_before_rxid_fix.csv"), index=False)

    if "RxID" not in rx.columns:
        rx["RxID"] = pd.NA
    rx["RxID"] = rx["RxID"].astype("string").str.strip()
    missing = rx["RxID"].isna() | (rx["RxID"] == "")

    date_col = "FillDate" if "FillDate" in rx.columns else ("RefillDate" if "RefillDate" in rx.columns else None)
    if date_col is None:
        raise ValueError("Need FillDate or RefillDate to generate stable RxIDs")

    rx[date_col] = pd.to_datetime(rx[date_col], errors="coerce")
    key = (
        rx["PatientID"].fillna("").astype("string").str.strip() + "|" +
        rx["DrugID"].fillna("").astype("string").str.strip() + "|" +
        rx[date_col].dt.strftime("%Y%m%d").fillna("00000000").astype("string")
    )
    h = pd.util.hash_pandas_object(key, index=False).astype("uint64")
    rx.loc[missing, "RxID"] = "RX" + h[missing].astype(str).str.zfill(20)

    rx.to_csv(p, index=False)
    print("Fixed RxID blanks:", int(missing.sum()), "-> now missing", int((rx["RxID"].isna() | (rx["RxID"]=="")).sum()))

def rebuild_sales_from_prescriptions(prescriptions_path="prescriptions.csv", patients_path="patients.csv", out_path="sales.csv"):
    """Rebuild sales.csv from prescriptions + patients (ensures RxID populated)."""
    rx = pd.read_csv(prescriptions_path, low_memory=False, dtype={"RxID":"string","DrugID":"string","PatientID":"string"})
    patients = pd.read_csv(patients_path, low_memory=False, dtype={"PatientID":"string"})

    if "InsuranceType" not in rx.columns:
        rx = rx.merge(patients[["PatientID","InsuranceType"]], on="PatientID", how="left")

    date_col = "FillDate" if "FillDate" in rx.columns else "RefillDate"
    rx[date_col] = pd.to_datetime(rx[date_col], errors="coerce")

    if "RxID" not in rx.columns:
        raise ValueError("RxID missing in prescriptions. Run fix_missing_rxid_in_prescriptions first.")

    rx["RxID"] = rx["RxID"].astype("string").str.strip()
    if (rx["RxID"].isna() | (rx["RxID"]=="")).any():
        raise ValueError("RxID still has blanks. Run fix_missing_rxid_in_prescriptions first.")

    rx["Revenue"] = pd.to_numeric(rx["Cost"], errors="coerce").fillna(0)

    sales = rx.dropna(subset=[date_col]).sort_values(date_col).reset_index(drop=True)
    sales["TransactionID"] = [f"T{str(i).zfill(8)}" for i in range(1, len(sales) + 1)]
    sales["Date"] = sales[date_col].dt.date.astype(str)
    sales["PaymentType"] = sales["InsuranceType"].fillna("Unknown").astype(str).str.strip()

    sales = sales[["TransactionID","DrugID","Date","PaymentType","Revenue","RxID"]]
    sales.to_csv(out_path, index=False)
    print("Rebuilt", out_path, "rows", len(sales))